## DATA 620 Project Three Instructions

- Split the Names Corpus into three subsets: 500 words for the test set, 500 words for the dev-test set, and the remaining 6900 words for the training set.

- Starting with the example name gender classifier, make incremental improvements.

- Use the dev-test set to check your progress.

- Check its final performance on the test set.

- How does the performance on the test set compare to the performance on the dev-test set?

- Is this what you'd expect?

### Executive Summary

Following the Bird et al (2009) chapter 6 guidance, a gender classification model was built using the NLTK names corpus and a Naive Bayes classifier, following a Supervised Learning workflow. The dataset of labeled names was split into training, development, and test sets, and feature engineering was applied to convert names into feature extractors based on known patterns. The model was trained on the training set and iteratively improved using the dev set. Using just the first letter of names, I started with a 64.6% performance rate on the training data and a 65.8% rate on the dev_set data. The model incrementally improved over three feature extractor improvements (77%, 80.4% and up to 82.2%). However, on unseen test set, performance was only 67%, indicating that it does not generalize well. I speculate that this is a case of overfitting. This could be caused by being too specific in the extraction features, as seen in the Slepian & Galinsky (2016) features, or by features involving the last three letters of the names. This outcome highlights the importance of proper data splitting, avoiding overfitting during feature design, and ensuring that evaluation on truly unseen data reflects real-world performance.

### MODULE 1: Feature Extractor.

The feature extractor builds a model (a function) that turns raw text (a word) into a labeled feature that a machine learning model or other program can use for analysis or prediction. According to the Bird et al. (2009) text, the feature extractor decides which features of the input are relevant and how to encode them. Bird et al. (2009) use the example of extracting the last letter of the word 'Shrek'. Below, I modified that example to show the feature extractor returning the value of the first letter of a name, 'Mary'. Note that the function strips case sensitivity.

In [1]:
def gender_features(word):
    word = word.lower().strip()
    
    return {
        'first_letter': word[0]}

gender_features('Mary')

{'first_letter': 'm'}

### MODULE 2: Load the names Data from the nltk library.

According to the Bird et al. (2009) text, the next step is to prepare a list of examples and corresponding class labels. The nltk library contains lists of known male and female names in text files. The labeled_names object is created, and it iterates over each name in names.words('male.txt'), and makes a pair (name, 'male') so every name gets labeled as ‘male’, and does the same for the females. The names are selected randomly and set to a seed for replicability.

In [2]:
import nltk
nltk.download('names', quiet=True)
from nltk.corpus import names
labeled_names = ([(name, 'male') for name in names.words('male.txt')] +
                 [(name, 'female') for name in names.words('female.txt')])
import random
random.seed(100)
random.shuffle(labeled_names)

### MODULE 3: Feature Set Creation.

According to the Bird et al. (2009) text, the next step is to create a feature set, which is a dictionary that maps from feature names to their values. The featuresets organize the "names" data into a format that the classifier can look for patterns. The featuresets object iterates over each (n, gender) pair in labeled_names, converts the name n into features using gender_features(n), and stores those features with the same gender.

In [3]:
featuresets = [(gender_features(n), gender) for (n, gender) in labeled_names]

for item in featuresets[:5]:
    print(item)

({'first_letter': 'p'}, 'male')
({'first_letter': 'b'}, 'male')
({'first_letter': 'p'}, 'female')
({'first_letter': 'f'}, 'female')
({'first_letter': 'c'}, 'male')


### MODULE 4: Split the Names Corpus: Training, Dev-Test, Test Subsets 

In the following module, I split the data into three subsets: 500 words for the test set, 500 words for the dev-test set, and the remaining 6900 words for the training set. The training set is used to train the model, and the dev-test set is used to perform error analysis. The test set serves in our final evaluation of the system.

In [4]:
test_names = labeled_names[:500]
dev_names = labeled_names[500:1000]
train_names = labeled_names[1000:]

### MODULE 5: Convert Data to Feature Sets

Each object below takes each name and turns it into features, then pairs those features with the correct gender so the model can learn from them.

In [5]:
train_set = [(gender_features(n), g) for (n, g) in train_names]
dev_set   = [(gender_features(n), g) for (n, g) in dev_names]
test_set  = [(gender_features(n), g) for (n, g) in test_names]

### MODULE 6: New 'naive Bayes' Classifier Creation Tested on Training Data

The training set is used to train a new "naive Bayes" classifier.

In [6]:
classifier = nltk.NaiveBayesClassifier.train(train_set)

print(f"{nltk.classify.accuracy(classifier, train_set):.3f}")

0.646


### MODULE 7: New 'naive Bayes' Classifier Creation Tested on Dev-Set Data

The code below takes the naive Bayes classifier, which is trained on the training data, then tested on the dev_set data.

In [7]:
print(nltk.classify.accuracy(classifier, dev_set))

0.658


### MODULE 8: Most Informative Features

In this module, I examine the classifier (tested on the training data) to determine which features it found most effective for distinguishing the names' genders. The interpretation is that if a male has a name that starts with a 'w', it is 4.3 times more likely to be a male over a female name. If it begins with a 'k', it is twice as likely to be a female name.

In [8]:
classifier.show_most_informative_features(5)

Most Informative Features
            first_letter = 'w'              male : female =      4.3 : 1.0
            first_letter = 'q'              male : female =      2.9 : 1.0
            first_letter = 'x'              male : female =      2.3 : 1.0
            first_letter = 'k'            female : male   =      2.3 : 1.0
            first_letter = 'h'              male : female =      2.1 : 1.0


### MODULE 9: Diagnose Errors

This output shows the predicted names versus the actual classifications from th dev_test corpus. For example, the model predicted Heidi to be a male name, but it is actually a female name. With understanding individual error cases where the model predicted the wrong label, I can try to determine what additional pieces of information would allow it to make the right decision, and adjust the feature set accordingly. 

In [9]:
errors = []
for (name, tag) in dev_names:
    guess = classifier.classify(gender_features(name))
    if guess != tag:
        errors.append( (tag, guess, name) )

for (tag, guess, name) in sorted(errors):
    print('correct={:<8} guess={:<8s} name={:<30}'.format(tag, guess, name))

correct=female   guess=male     name=Heidi                         
correct=female   guess=male     name=Helaina                       
correct=female   guess=male     name=Helyn                         
correct=female   guess=male     name=Hermina                       
correct=female   guess=male     name=Hermine                       
correct=female   guess=male     name=Hester                        
correct=female   guess=male     name=Hinda                         
correct=female   guess=male     name=Winifred                      
correct=female   guess=male     name=Wynnie                        
correct=female   guess=male     name=Yoko                          
correct=male     guess=female   name=Addie                         
correct=male     guess=female   name=Albrecht                      
correct=male     guess=female   name=Aleck                         
correct=male     guess=female   name=Alessandro                    
correct=male     guess=female   name=Alston     

### MODULE 10: Gender classifier improvement #1.

In the Bird et al. (2009) text, they discuss classifying names ending in a, e and i to be likely associated with females, while names ending in k, o, r, s and t are likely to be associated with males. This is given in the text.

In [10]:
def gender_features(word):
    word = word.lower().strip()
    
    return {
        'ends_with_aeiy': word[-1] in 'aeiy',
        'ends_with_korst': word[-1] in 'korst',
        'ends_with_yn': word[-2] in 'yn'
    }
print(gender_features("Mary"))

{'ends_with_aeiy': True, 'ends_with_korst': False, 'ends_with_yn': False}


In [11]:
train_set = [(gender_features(n), gender) for (n, gender) in train_names]
dev_set = [(gender_features(n), gender) for (n, gender) in dev_names]
classifier = nltk.NaiveBayesClassifier.train(train_set)
print(nltk.classify.accuracy(classifier, dev_set))

0.77


In [12]:
classifier.show_most_informative_features(5)

Most Informative Features
         ends_with_korst = True             male : female =      5.9 : 1.0
          ends_with_aeiy = False            male : female =      3.5 : 1.0
            ends_with_yn = True           female : male   =      2.8 : 1.0
          ends_with_aeiy = True           female : male   =      2.7 : 1.0
         ends_with_korst = False          female : male   =      1.3 : 1.0


In [13]:
errors = []
for (name, tag) in dev_names:
    guess = classifier.classify(gender_features(name))
    if guess != tag:
        errors.append( (tag, guess, name) )

for (tag, guess, name) in sorted(errors):
    print('correct={:<8} guess={:<8s} name={:<30}'.format(tag, guess, name))

correct=female   guess=male     name=Abagail                       
correct=female   guess=male     name=Abigael                       
correct=female   guess=male     name=Adriaens                      
correct=female   guess=male     name=Alis                          
correct=female   guess=male     name=Allis                         
correct=female   guess=male     name=Alys                          
correct=female   guess=male     name=Anabel                        
correct=female   guess=male     name=Anais                         
correct=female   guess=male     name=Arabel                        
correct=female   guess=male     name=Arleen                        
correct=female   guess=male     name=Ashleigh                      
correct=female   guess=male     name=Avrit                         
correct=female   guess=male     name=Barb                          
correct=female   guess=male     name=Clair                         
correct=female   guess=male     name=Con        

### MODULE 11: Gender classifier improvement #3.

The following improvements address the errors and look for possible two and three last letters patterns, the first letter and length of the names. 

In [14]:
def gender_features(word):
    word = word.lower()
    return {
        'last_letter': word[-1:],
        'last_two_letters': word[-2:],
        'last_three_letters': word[-3:],
        'first_letter': word[0],
        'length': len(word),
    }
print(gender_features("Mary"))

{'last_letter': 'y', 'last_two_letters': 'ry', 'last_three_letters': 'ary', 'first_letter': 'm', 'length': 4}


In [15]:
train_set = [(gender_features(n), gender) for (n, gender) in train_names]
dev_set = [(gender_features(n), gender) for (n, gender) in dev_names]
classifier = nltk.NaiveBayesClassifier.train(train_set)
print(nltk.classify.accuracy(classifier, dev_set))

0.804


In [16]:
classifier.show_most_informative_features(5)

Most Informative Features
        last_two_letters = 'na'           female : male   =     92.8 : 1.0
        last_two_letters = 'ia'           female : male   =     88.7 : 1.0
        last_two_letters = 'la'           female : male   =     71.0 : 1.0
             last_letter = 'a'            female : male   =     34.9 : 1.0
        last_two_letters = 'sa'           female : male   =     34.8 : 1.0


In [17]:
errors = []
for (name, tag) in dev_names:
    guess = classifier.classify(gender_features(name))
    if guess != tag:
        errors.append( (tag, guess, name) )

for (tag, guess, name) in sorted(errors):
    print('correct={:<8} guess={:<8s} name={:<30}'.format(tag, guess, name))

correct=female   guess=male     name=Abagail                       
correct=female   guess=male     name=Abigael                       
correct=female   guess=male     name=Adriaens                      
correct=female   guess=male     name=Aime                          
correct=female   guess=male     name=Alis                          
correct=female   guess=male     name=Allis                         
correct=female   guess=male     name=Anais                         
correct=female   guess=male     name=Ashleigh                      
correct=female   guess=male     name=Aubrey                        
correct=female   guess=male     name=Barb                          
correct=female   guess=male     name=Berry                         
correct=female   guess=male     name=Blakeley                      
correct=female   guess=male     name=Carley                        
correct=female   guess=male     name=Clair                         
correct=female   guess=male     name=Con        

### MODULE 12: Gender classifier improvement #4.

To make several improvements to the gender classifier, I consulted an article published by Slepian & Galinsky (2016) in the Journal of Personality and Social Psychology. They published an article, [***The Voiced Pronunciation of Initial Phonemes Predicts the Gender of Names***](https://doi-org.remote.baruch.cuny.edu/10.1037/pspa0000041). The authors explored the systematic effect of phonetic qualities of given names by investigating the link between sound symbolism and gender. They proposed that vocal cord vibration during the pronunciation of an initial phoneme plays a critical role in explaining which names are assigned to males versus females. This produces a voiced gendered name effect, whereby voiced phonemes (vibration of the vocal cords) are more associated with male names, and unvoiced phonemes (no vibration of the vocal cords) are more associated with female names. 

The article gives examples of how one can distinguish between a voiced or unvoiced sound. To experience this difference, the authors suggest the reader pronounce the words “this” and “thin” aloud while placing a finger on the laryngeal prominence (i.e., the “Adam’s apple”). The "th" sound in this is voiced, whereas the "th" sound in thin is unvoiced. Or, pronounce the words “bear” and “pear.” One should notice in both examples (during the first phoneme) a vibration present in the former words (“this,” “bear”), but not the latter words (“thin,” “pear”), with the initial phonemes in the latter words sounding more breathy.

In Chapter 4.2 of the Bird et al text, they discuss that the NLTK library includes the CMU Pronouncing Dictionary for US English. This appears to be a bit of a complex process, so I opted to follow the Slepian & Galinsky method to distinguish voiced v. unvoiced sounds. I went through the alphabet, pronouncing each letter to classify it as voiced or unvoiced. 

I modified the feature extractions to now learn from the likely female or male last letter of the name, and the voiced or unvoiced likely male or female last letter of the name.

The output suggests that the most informative features are that names ending in "na" or "ia" are nearly 9 times more likely to be female. This classifier also increased performance on the dev_set to 82.2%, up from 80.4% without the Slepian & Galinsky (2016) features.

In [18]:
def gender_features(word):
    word = word.lower()
    
    return {
        'voiced_ends_with_bdgvzjlmnryw': word[-1] in 'bdgvzjlmnryw',
        'unvoiced_end_with_ptkfsh': word[-1] in 'ptkfsh',
        'last_two_letters': word[-2:],
        'last_three_letters': word[-3:],
        'first_letter': word[0],
        'length': len(word),
    }
print(gender_features("Mary"))

{'voiced_ends_with_bdgvzjlmnryw': True, 'unvoiced_end_with_ptkfsh': False, 'last_two_letters': 'ry', 'last_three_letters': 'ary', 'first_letter': 'm', 'length': 4}


In [19]:
train_set = [(gender_features(n), gender) for (n, gender) in train_names]
dev_set = [(gender_features(n), gender) for (n, gender) in dev_names]
classifier = nltk.NaiveBayesClassifier.train(train_set)
print(nltk.classify.accuracy(classifier, dev_set))

0.822


In [20]:
classifier.show_most_informative_features(5)

Most Informative Features
        last_two_letters = 'na'           female : male   =     92.8 : 1.0
        last_two_letters = 'ia'           female : male   =     88.7 : 1.0
        last_two_letters = 'la'           female : male   =     71.0 : 1.0
        last_two_letters = 'sa'           female : male   =     34.8 : 1.0
        last_two_letters = 'ra'           female : male   =     34.2 : 1.0


In [21]:
errors = []
for (name, tag) in dev_names:
    guess = classifier.classify(gender_features(name))
    if guess != tag:
        errors.append( (tag, guess, name) )

for (tag, guess, name) in sorted(errors):
    print('correct={:<8} guess={:<8s} name={:<30}'.format(tag, guess, name))

correct=female   guess=male     name=Abagail                       
correct=female   guess=male     name=Abigael                       
correct=female   guess=male     name=Adriaens                      
correct=female   guess=male     name=Aime                          
correct=female   guess=male     name=Alis                          
correct=female   guess=male     name=Allis                         
correct=female   guess=male     name=Ashleigh                      
correct=female   guess=male     name=Aubrey                        
correct=female   guess=male     name=Barb                          
correct=female   guess=male     name=Berry                         
correct=female   guess=male     name=Blakeley                      
correct=female   guess=male     name=Carey                         
correct=female   guess=male     name=Carley                        
correct=female   guess=male     name=Clair                         
correct=female   guess=male     name=Con        

### MODULE 13: Test Set Performance.

The code below evaluates the test set. The results are a 67% performance rate. This is unexpected. It is a significant drop from the dev_set. I speculate that this is a case of overfitting. This could be caused by being too specific in the extraction features, such as we see in the Slepian & Galinsky (2016) features, or possibly features involving the last three letters of the names.

In [22]:
print(nltk.classify.accuracy(classifier, test_set))

0.67
